In [2]:
import os
import requests
import dotenv

dotenv.load_dotenv(
    dotenv_path="../../.env"
)

token = os.environ.get("GITHUB_TOKEN")
headers = {
    "Authorization": f"Bearer {token}",
    "Accept": "application/vnd.github+json"
}

url = "https://api.github.com/user/repos"
params = {
    "visibility": "all",
    "affiliation": "owner,collaborator,organization_member",
    "per_page": 100,
    "page": 1
}

repos = []

while True:
    r = requests.get(url, headers=headers, params=params)
    r.raise_for_status()
    data = r.json()
    if not data:
        break
    repos.extend(data)
    params["page"] += 1

for repo in repos:
    print(f"{repo['full_name']} | private={repo['private']} | permissions={repo['permissions']}")


amcspsgtech/alumni-nomination-frontend | private=True | permissions={'admin': False, 'maintain': True, 'push': True, 'triage': True, 'pull': True}
amcspsgtech/alumni-nominations-backend | private=True | permissions={'admin': True, 'maintain': True, 'push': True, 'triage': True, 'pull': True}
CSA-Tech-Team/.github | private=True | permissions={'admin': False, 'maintain': False, 'push': False, 'triage': False, 'pull': True}
CSA-Tech-Team/404-not-found | private=True | permissions={'admin': False, 'maintain': False, 'push': False, 'triage': False, 'pull': True}
CSA-Tech-Team/alumini-meet-frontend | private=False | permissions={'admin': False, 'maintain': False, 'push': True, 'triage': True, 'pull': True}
CSA-Tech-Team/alumni-meet-backend | private=False | permissions={'admin': True, 'maintain': True, 'push': True, 'triage': True, 'pull': True}
CSA-Tech-Team/axios | private=True | permissions={'admin': False, 'maintain': False, 'push': False, 'triage': False, 'pull': True}
CSA-Tech-Team/ax

In [3]:
import os
import requests
from fastapi import FastAPI, Request
from fastapi.responses import RedirectResponse

app = FastAPI()

CLIENT_ID = os.environ["GITHUB_CLIENT_ID"]
CLIENT_SECRET = os.environ["GITHUB_CLIENT_SECRET"]
REDIRECT_URI = os.environ["GITHUB_REDIRECT_URI"]

@app.get("/login/github")
def login_github():
    auth_url = (
        "https://github.com/login/oauth/authorize"
        f"?client_id={CLIENT_ID}"
        f"&redirect_uri={REDIRECT_URI}"
        "&scope=repo%20read:org"
    )
    return RedirectResponse(auth_url)

@app.get("/callback/github")
def callback_github(request: Request):
    code = request.query_params.get("code")
    token_resp = requests.post(
        "https://github.com/login/oauth/access_token",
        headers={"Accept": "application/json"},
        data={
            "client_id": CLIENT_ID,
            "client_secret": CLIENT_SECRET,
            "code": code,
            "redirect_uri": REDIRECT_URI,
        },
    )
    token = token_resp.json().get("access_token")
    headers = {"Authorization": f"Bearer {token}", "Accept": "application/vnd.github+json"}
    repos = []
    page = 1
    while True:
        r = requests.get(
            "https://api.github.com/user/repos",
            headers=headers,
            params={"visibility": "all", "affiliation": "owner,collaborator,organization_member", "page": page, "per_page": 100},
        )
        data = r.json()
        if not data:
            break
        repos.extend(data)
        page += 1
    for repo in repos:
        print(repo["full_name"])
    return {"status": "completed", "repos_count": len(repos)}

In [5]:
if __name__ == "__main__":
    import uvicorn
    uvicorn.run(app, host="0.0.0.0", port=8000)

RuntimeError: asyncio.run() cannot be called from a running event loop

In [4]:
import sys
print(sys.executable)

/home/aklamaash/Desktop/launchpad/notebook/venv/bin/python
